# AI-Based Disease Risk Prediction & Explainability System
**Author:** Balaji  
**Environment:** Google Colab  
**Dataset:** UCI Heart Disease (Cleveland subset)  
**Stack:** Python, pandas, scikit-learn, XGBoost, SHAP

## Project overview
This educational machine-learning workflow estimates whether a record in the UCI Heart Disease dataset belongs to the dataset's heart-disease-positive class. It demonstrates data validation, preprocessing, model comparison, held-out evaluation, explainability and a single-record demonstration.

> **Important:** This is not a medical device, screening tool or diagnosis. It must not be used to make clinical decisions. Predictions are dataset-specific and require independent clinical validation before any real-world use.

### Workflow
1. Load the public UCI dataset and inspect it.
2. Clean missing values and convert the target to a binary label (`0` = no disease recorded; `1` = disease recorded).
3. Split into train/test sets before fitting preprocessing.
4. Train Logistic Regression, Random Forest and XGBoost (where available).
5. Compare models on held-out data; report accuracy, precision, recall, F1 and ROC-AUC.
6. Explain the selected model with SHAP where supported.
7. Demonstrate prediction for one held-out row and export artifacts/reports.

## 1. Install dependencies

In [ ]:
%pip -q install ucimlrepo xgboost shap joblib

## 2. Imports and reproducibility

In [ ]:
import os, json, warnings, joblib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from IPython.display import display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("artifacts", exist_ok=True)
os.makedirs("reports", exist_ok=True)

## 3. Load UCI Heart Disease data

In [ ]:
# UCI Machine Learning Repository dataset ID 45 (Heart Disease)
heart = fetch_ucirepo(id=45)
X_raw = heart.data.features.copy()
y_raw = heart.data.targets.iloc[:, 0].copy()

display(X_raw.head())
print("Feature shape:", X_raw.shape)
print("Target values:", pd.Series(y_raw).value_counts(dropna=False).to_dict())
print("Metadata:", heart.metadata.get("name", "UCI Heart Disease"))

## 4. Validate and prepare data
The original target uses values 0 through 4. For this demonstration, values greater than zero are grouped into a positive class. This binary grouping is a project design choice, not a clinical definition.

In [ ]:
# Normalize common missing-value markers and coerce numeric columns where possible.
X = X_raw.replace("?", np.nan).copy()
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

y_numeric = pd.to_numeric(y_raw, errors="coerce")
valid = y_numeric.notna()
X = X.loc[valid].reset_index(drop=True)
y = (y_numeric.loc[valid].astype(int) > 0).astype(int).reset_index(drop=True)

print("Clean feature shape:", X.shape)
print("Missing values by column:")
display(X.isna().sum().to_frame("missing_count"))
print("Binary target distribution:")
display(y.value_counts().rename(index={0:"No disease recorded", 1:"Disease recorded"}))
if y.nunique() != 2:
    raise ValueError("Expected two target classes after binary conversion.")

## 5. Exploratory data analysis

In [ ]:
display(X.describe().T)
ax = y.value_counts().sort_index().plot(kind="bar", rot=0)
ax.set_xticks([0,1], ["Class 0", "Class 1"])
ax.set_ylabel("Records")
ax.set_title("Target class distribution")
plt.tight_layout(); plt.show()

missing_pct = X.isna().mean().sort_values(ascending=False) * 100
print("Missingness (%):")
display(missing_pct.to_frame("missing_percent"))

## 6. Train/test split and preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
num_cols = X_train.columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols)
], remainder="drop")
print("Train:", X_train.shape, "Test:", X_test.shape)

## 7. Train and compare models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
}
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(n_estimators=350, max_depth=3, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.85, reg_lambda=2,
        eval_metric="logloss", random_state=RANDOM_STATE)
except Exception as e:
    print("XGBoost unavailable; continuing with scikit-learn models:", e)

pipelines, rows = {}, []
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba) if y_test.nunique() == 2 else np.nan
    }
    rows.append(metrics); pipelines[name] = pipe
results = pd.DataFrame(rows).sort_values(["roc_auc", "f1"], ascending=False)
display(results)
results.to_csv("reports/model_metrics.csv", index=False)
best_name = results.iloc[0]["model"]
best_model = pipelines[best_name]
print("Selected by held-out ROC-AUC then F1:", best_name)

## 8. Held-out evaluation

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred, target_names=["Class 0", "Class 1"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Class 0", "Class 1"])
plt.title(f"Confusion Matrix — {best_name}"); plt.tight_layout(); plt.show()
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title(f"ROC Curve — {best_name}"); plt.tight_layout(); plt.show()

## 9. Model explainability with SHAP

In [ ]:
# SHAP support varies by model/version. If TreeExplainer is unsupported, use permutation importance.
try:
    import shap
    Xt = best_model.named_steps["preprocess"].transform(X_test)
    estimator = best_model.named_steps["model"]
    if hasattr(estimator, "feature_importances_"):
        explainer = shap.TreeExplainer(estimator)
        shap_values = explainer.shap_values(Xt)
        if isinstance(shap_values, list):
            shap_values = shap_values[-1]
        if getattr(shap_values, "ndim", 0) == 3:
            shap_values = shap_values[:, :, -1]
        shap.summary_plot(shap_values, Xt, feature_names=num_cols, show=True)
    else:
        print("SHAP TreeExplainer is intended for tree models; using permutation importance below.")
except Exception as e:
    print("SHAP explanation could not be generated in this runtime:", repr(e))
    print("Permutation importance is calculated in the next cell as a fallback.")

## 10. Permutation importance (model-agnostic fallback)

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=20,
                              random_state=RANDOM_STATE, scoring="roc_auc")
importance = pd.DataFrame({"feature": X.columns,
                           "importance_mean": perm.importances_mean,
                           "importance_std": perm.importances_std}).sort_values("importance_mean", ascending=False)
display(importance)
importance.to_csv("reports/permutation_importance.csv", index=False)
importance.head(15).sort_values("importance_mean").plot(
    x="feature", y="importance_mean", kind="barh", legend=False,
    title="Top permutation importances")
plt.xlabel("Mean decrease in ROC-AUC after permutation")
plt.tight_layout(); plt.show()

## 11. Demonstrate a prediction on one held-out record

In [ ]:
sample = X_test.iloc[[0]].copy()
probability = float(best_model.predict_proba(sample)[0, 1])
prediction = int(best_model.predict(sample)[0])
print("Predicted dataset class:", prediction)
print(f"Model score for positive dataset class: {probability:.3f}")
display(sample)
print("Interpretation: this is a model output for a dataset record, not a diagnosis or an estimate of an individual's clinical risk.")

## 12. Save model and run metadata

In [ ]:
joblib.dump(best_model, "artifacts/best_disease_risk_pipeline.joblib")
metadata = {
    "project": "AI-Based Disease Risk Prediction & Explainability System",
    "author": "Balaji",
    "dataset": "UCI Heart Disease (dataset ID 45)",
    "positive_class_definition": "original target > 0",
    "selected_model": best_name,
    "test_metrics": results.iloc[0].to_dict(),
    "random_state": RANDOM_STATE,
    "warning": "Educational prototype only; not for diagnosis or clinical decisions."
}
with open("reports/run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=float)
print("Saved artifacts and reports.")
print("Download the artifacts/reports folders from the Colab Files panel if needed.")

## 13. Limitations and responsible use
- The dataset is small, historical and not representative of all populations or current clinical practice.
- A random train/test split is not external validation; reported metrics can vary by split.
- The binary target collapses original severity categories and loses information.
- Missing-value imputation and model explanations do not establish causality.
- Dataset class probabilities are not calibrated individual clinical risks.
- Do not use this notebook for diagnosis, treatment, triage, insurance, employment or other consequential decisions.
- Any clinical application requires qualified medical oversight, representative prospective validation, privacy/security review and regulatory assessment.

## Author

**Balaji**  
Educational machine-learning portfolio project.